## Choose either dataset to test binary classification. Create a 4-layer Dense network, 
## test with minimum neurons (64, 32, 16, 1), then Increase Units in Dense layers and 
## also Increase Number of Layers add BatchNormalization to Layers add Dropout after Layers.

# Titanic Dataset
https://www.kaggle.com/datasets/vicsuperman/prediction-of-music-genre?select=music_genre.csv

In [112]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder 
from sklearn.model_selection import train_test_split

print("ok")

ok


In [113]:
data = pd.read_csv("Titanic-Dataset.csv")
data.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [114]:
data.drop(['Cabin', 'Fare', 'Name', 'PassengerId', 'Ticket'] , axis = 1 , inplace = True)

In [115]:
data.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Embarked      2
dtype: int64

In [116]:
data = data.fillna(data.median(numeric_only=True))

In [117]:
data.dropna(inplace = True)

In [118]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  889 non-null    int64  
 1   Pclass    889 non-null    int64  
 2   Sex       889 non-null    object 
 3   Age       889 non-null    float64
 4   SibSp     889 non-null    int64  
 5   Parch     889 non-null    int64  
 6   Embarked  889 non-null    object 
dtypes: float64(1), int64(4), object(2)
memory usage: 55.6+ KB


In [119]:
# Label encode Object Datatypes 
data_types_dict = dict(data.dtypes)
# Keep track Mappings Column name to LabelEncoders
Label_encoder_collection = {}
for col_name , data_type in data_types_dict.items():
    if data_type == 'object':
        Le = LabelEncoder()
        data[col_name] = Le.fit_transform(data[col_name])
        Label_encoder_collection[col_name] = Le 

data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  889 non-null    int64  
 1   Pclass    889 non-null    int64  
 2   Sex       889 non-null    int64  
 3   Age       889 non-null    float64
 4   SibSp     889 non-null    int64  
 5   Parch     889 non-null    int64  
 6   Embarked  889 non-null    int64  
dtypes: float64(1), int64(6)
memory usage: 55.6 KB


In [120]:
for col_name, label_encoder in Label_encoder_collection.items():
    print(f"Label Encoding mapping for column: {col_name}")
    # Get the mapping (original labels to encoded integers)
    mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
    print(mapping)
    print("-" * 30)

Label Encoding mapping for column: Sex
{'female': 0, 'male': 1}
------------------------------
Label Encoding mapping for column: Embarked
{'C': 0, 'Q': 1, 'S': 2}
------------------------------


In [121]:
num_genres = 1

In [122]:
# move workout type to end
col_to_move = 'Survived'
cols = [col for col in data.columns if col != col_to_move] + [col_to_move]
data = data[cols]

In [123]:
data.head(5)

,Pclass,Sex,Age,SibSp,Parch,Embarked,Survived
0,3,1,22.0,1,0,2,0
1,1,0,38.0,1,0,0,1
2,3,0,26.0,0,0,2,1
3,1,0,35.0,1,0,2,1
4,3,1,35.0,0,0,2,0


In [124]:
X = data.iloc[: , :-1].values.astype(np.float32)
Y = data.iloc[: , -1].values.astype(int)

In [125]:
X

array([[ 3.,  1., 22.,  1.,  0.,  2.],
       [ 1.,  0., 38.,  1.,  0.,  0.],
       [ 3.,  0., 26.,  0.,  0.,  2.],
       ...,
       [ 3.,  0., 28.,  1.,  2.,  2.],
       [ 1.,  1., 26.,  0.,  0.,  0.],
       [ 3.,  1., 32.,  0.,  0.,  1.]], dtype=float32)

In [126]:
Y

array([0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1,
       0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0,
       1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1,
       0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0,
       0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1,
       0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,

In [127]:
Xtrain , Xtest , Ytrain , Ytest = train_test_split(X, Y, test_size = 0.3, random_state = 333)

In [128]:
# normalizare
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
Xtrain = scaler.fit_transform(Xtrain)
Xtest = scaler.transform(Xtest)

In [129]:
input_shape = Xtrain.shape[1]

# model = tf.keras.models.Sequential()
# model.add(tf.keras.Input(shape=(input_shape,)))
# model.add(tf.keras.layers.Dense(128)
# model.add(tf.keras.layers.Dense(64)
# model.add(tf.keras.layers.Dense(32)
# model.add(tf.keras.layers.Dense(num_genres, activation='sigmoid'))  # pentru clasificare multi-class

# # print summary to undertstand your neural network flow
# model.summary()

In [130]:
Optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(Optimizer, loss='sparse_categorical_crossentropy', metrics=["accuracy"])

In [131]:
history = model.fit(Xtrain , Ytrain , validation_data=(Xtest , Ytest), epochs=20)

Epoch 1/20


/home/catalin/.local/lib/python3.9/site-packages/keras/src/ops/nn.py:908: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
2025-04-17 17:47:26.301913: W tensorflow/core/framework/op_kernel.cc:1857] OP_REQUIRES failed at sparse_xent_op.cc:103 : INVALID_ARGUMENT: Received a label value of 1 which is outside the valid range of [0, 1).  Label values: 0 1 1 1 0 0 1 1 0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0


InvalidArgumentError: Graph execution error:

Detected at node compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/SparseSoftmaxCrossEntropyWithLogits defined at (most recent call last):
  File "/usr/lib64/python3.9/runpy.py", line 197, in _run_module_as_main

  File "/usr/lib64/python3.9/runpy.py", line 87, in _run_code

  File "/home/catalin/.local/lib/python3.9/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/catalin/.local/lib/python3.9/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/catalin/.local/lib/python3.9/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/catalin/.local/lib/python3.9/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib64/python3.9/asyncio/base_events.py", line 601, in run_forever

  File "/usr/lib64/python3.9/asyncio/base_events.py", line 1905, in _run_once

  File "/usr/lib64/python3.9/asyncio/events.py", line 

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

# "Loss Plot"
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

In [ ]:
predictions = np.argmax(model.predict(Xtest), axis=1)

In [ ]:
from sklearn.metrics import accuracy_score , confusion_matrix
accuracy_score(Ytest , predictions) # Accuracy Seems good

In [ ]:
confusion_matrix(Ytest , predictions)

In [ ]:
from sklearn.metrics import classification_report 
print(classification_report(Ytest, predictions, zero_division=0))

In [ ]:
# we will try to improve our model's performance 
# ----------- Add class weights when Data is Imbalanced -----------------------
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(class_weight = 'balanced',  classes = np.unique(Ytrain), y=Ytrain)

model_class_weights = {}
for e , weight in enumerate(class_weights):
    model_class_weights[e] = weight

In [ ]:
model_class_weights

In [ ]:
history = model.fit(Xtrain , Ytrain , validation_data=(Xtest , Ytest) , epochs=20, class_weight=model_class_weights)

In [ ]:
predictions = np.argmax(model.predict(Xtest), axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(Ytest, predictions)

In [ ]:
Xtrain , Xtest , Ytrain , Ytest = train_test_split(X, Y, test_size = 0.3, random_state = 333)

In [ ]:
# normalizare
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
Xtrain = scaler.fit_transform(Xtrain)
Xtest = scaler.transform(Xtest)

In [ ]:
# Increase Units in Dense layers and Also Increase Number of Layers 
# Add BatchNormalization to Layers 
# Add Dropout after Layers 

model = tf.keras.models.Sequential()
model.add(tf.keras.Input(shape=(input_shape,)))

model.add(tf.keras.layers.Dense(256 , activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dropout(0.3))

model.add(tf.keras.layers.Dense(128 , activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dropout(0.3))

model.add(tf.keras.layers.Dense(64 , activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dropout(0.2))

model.add(tf.keras.layers.Dense(num_genres , activation='softmax'))

# model = tf.keras.models.Sequential()
# model.add(tf.keras.Input(shape=(input_shape,)))
# model.add(tf.keras.layers.Dense(128, activation='relu'))
# model.add(tf.keras.layers.Dense(64, activation='relu'))
# model.add(tf.keras.layers.BatchNormalization())
# model.add(tf.keras.layers.Dense(32, activation='relu'))
# model.add(tf.keras.layers.Dense(num_genres, activation='softmax'))  # pentru clasificare multi-class

# print summary to undertstand your neural network flow
# model.summary()

Optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
# Optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)
model.compile(Optimizer, loss='sparse_categorical_crossentropy', metrics=["accuracy"])

history = model.fit(Xtrain , Ytrain , validation_data=(Xtest , Ytest) , epochs=80)

In [ ]:
Optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(Optimizer, loss='sparse_categorical_crossentropy', metrics=["accuracy"])

In [ ]:
history = model.fit(Xtrain , Ytrain , validation_data=(Xtest , Ytest) , epochs=20)

In [ ]:
#  "Accuracy Plot"
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

# "Loss Plot"
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

# We see graphs converging better compared to previous implementation

In [ ]:
# collect predictions 
predictions = np.argmax(model.predict(Xtest), axis=1)

In [ ]:
# Check Accuracy
accuracy_score(Ytest , predictions) # Accuracy is good

In [ ]:
# We don't see the problem of predicting everything as class 0
confusion_matrix(Ytest , predictions)

In [ ]:
print(classification_report(Ytest, predictions))